In [2]:
import graph_tool.all as gt

from utils.Flow import *

import time
import logging
import os

In [3]:
new_input_graphs = [
    'academia_edu',
    'amazon_copurchases/302',
    'anybeat',
    'arxiv_authors/CondMat',
    'arxiv_citation/HepPh',
    'as_skitter',
    'baidu',
    'berkstan_web',
    'caida_as/20071112',
    'chicago_road',
    'citeseer',
    'cora',
    'dblp_cite',
    'dblp_coauthor_snap',
    # 'dbpedia_link', ## Way to big
    'douban',
    'ego_social/gplus_combined',
    'email_enron',
    'email_eu',
    'epinions_trust',
    'flickr_aminer',
    'flickr_growth',
    'flixster',
    'foursquare_friendships/new',
    'gnutella/31',
    'google',
    'google_plus',
    'google_web',
    'hyves',
    'inploid',
    'internet_as',
    'lastfm_aminer',
    'linux',
    'livejournal',
    'livemocha',
    'marker_cafe',
    'marvel_universe',
    'mislove_osn/youtube',
    'myspace_aminer',
    'notre_dame_web',
    'openstreetmap/01-AL-counties-street_networks:01073_Jefferson_County', ## weight needs to be parsed
    'petster',
    'pgp_strong',
    'pokec',
    'python_dependency',
    'roadnet/CA',
    'scotus_majority/2008',
    # 'soc_net_comms', # - seems like repettition
    'social_location/brightkite',
    'stanford_web',
    'trec_web',
    'tree-of-life/9606',
    'twitter',
    'twitter_15m',
    # 'twitter_2009', # - really big network
    'twitter_sample',
    # 'twitter_social', # - too big
    'us_patents',
    # 'wiki_users', # -make positive weights
    # 'wikiconflict', # - make weight positive
    'wikipedia-en-talk',
    'wikipedia_growth',
    'wikipedia_link/az',
    'wikitree',
    'wordnet',
    'yahoo_ads',
    'sp_infectious',
    'genetic_multiplex/Homo',
    ('arxiv_collab/cond-mat-2005', 'value'),
    'prosper',
    'wiki_link_dyn',
    ('mist/ppi_interolog_worm', 'Count_paper'),
    'libimseti',
    'twitter_higgs/reply',
    'dblp_coauthor',
    'twitter_events/NYClimateMarch2014',
    'qa_user/askubuntu_all',
    'mag_history_coauthor/full-proj',
    'mag_geology_coauthor/full-proj',
    'wiki_talk/de',
    'dbpedia_all',
    'dblp_simplices/full-proj',
    ('bitcoin', 'count'),
    'word_adjacency/spanish',
    'facebook_wall',
    'digg_reply',
    'foldoc',
    'fly_hemibrain',
    'lkml_reply',
    'cofe',
    'slashdot_threads',
    'word_assoc',
    'human_brains/BNU1_0025915_2_DTI_DS16784',
    ('us_agencies/aggregate', 'link_counts'),
    'physics_collab/arXiv',
    'topology',
    'us_roads/DE'
]

In [4]:
def load_and_run_drawing(graph_path, path, is_mst):
    graph = gt.load_graph(graph_path)

    output_name = "mst_draw.png" if is_mst else "largest_component.png"

    gt.graph_draw(graph, output=f'{path}/{output_name}')

In [5]:
def get_graph_and_prepare_drawing(graph_name):
    try:
        path = prepare_folder(graph_name)

        file_path = get_network_file_path(graph_name, path, False)

        if os.path.isfile(file_path):
            in_separate_process(
                run=load_and_run_drawing,
                withArgs=(file_path, path, False),
                log_as=f'GC of {graph_name}'
            )
        else:
            logging.error(f'No GC graph file for {graph_name}')

        mst_path = get_network_file_path(graph_name, path, True)

        if os.path.isfile(mst_path):
            in_separate_process(
                run=load_and_run_drawing,
                withArgs=(mst_path, path, True),
                log_as=f'MST of {graph_name}'
            )
        else:
            logging.error(f'No MST graph file for {graph_name}')

    except Exception as ex:
        logging.error(f'failed to run: {graph_name} with {ex}')

In [6]:
def start_pipeline(data):
    config_logging(into='drawing_collection.log')

    logging.info('Starting drawing pipeline')
    start_time = time.time()

    for graph_data in data:
        if (isinstance(graph_data, tuple) and len(graph_data) >= 2):
            in_separate_process(
                run=get_graph_and_prepare_drawing,
                withArgs=(graph_data[0],),
                log_as=graph_data[0]
            )
        else:
            in_separate_process(
                run=get_graph_and_prepare_drawing,
                withArgs=(graph_data,),
                log_as=graph_data
            )

    end_time = time.time()

    compute_time = end_time - start_time

    readable_time = time.strftime("%H:%M:%S", time.gmtime(compute_time))

    logging.info(
        f'Drawing collection finished at {end_time} and took: {readable_time}')

    return compute_time

In [ ]:
start_pipeline(
    new_input_graphs
)